In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("julienjta/nyc-taxi-traffic")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\xz159\.cache\kagglehub\datasets\julienjta\nyc-taxi-traffic\versions\1


In [2]:
from glob import glob
import pandas as pd

fname = glob(path + "/*.csv")[0]
df = pd.read_csv(fname)

In [3]:
def overview(df, timestamp_col='timestamp'):
    # 转换为datetime类型
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors='coerce')

    print('\nDate Range:\n')
    print('Start:\t', df[timestamp_col].min())
    print('End:\t', df[timestamp_col].max())

    delta = df[timestamp_col].max() - df[timestamp_col].min()
    print('Days:\t', delta.days)


In [4]:
overview(df, timestamp_col='timestamp')


Date Range:

Start:	 2014-07-01 00:00:00
End:	 2015-01-31 23:30:00
Days:	 214


In [5]:
# A variety of resamples which I may or may not use
df_hourly = df.set_index('timestamp').resample('H').mean().reset_index()
df_daily = df.set_index('timestamp').resample('D').mean().reset_index()
df_weekly = df.set_index('timestamp').resample('W').mean().reset_index()

C:\Users\xz159\AppData\Local\Temp\ipykernel_9520\335819791.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.set_index('timestamp').resample('H').mean().reset_index()


In [6]:
# New features 
# Loop to cycle through both DataFrames
for DataFrame in [df_hourly, df_daily]:
    DataFrame['Weekday'] = (pd.Categorical(DataFrame['timestamp'].dt.strftime('%A'),
                                           categories=['Monday', 'Tuesday', 'Wednesday', 'Thursday','Friday', 'Saturday', 'Sunday'])
                           )
    DataFrame['Hour'] = DataFrame['timestamp'].dt.hour
    DataFrame['Day'] = DataFrame['timestamp'].dt.weekday
    DataFrame['Month'] = DataFrame['timestamp'].dt.month
    DataFrame['Year'] = DataFrame['timestamp'].dt.year
    DataFrame['Month_day'] = DataFrame['timestamp'].dt.day
    DataFrame['Lag'] = DataFrame['value'].shift(1)
    DataFrame['Rolling_Mean'] = DataFrame['value'].rolling(7, min_periods=1).mean()
    DataFrame = DataFrame.dropna()

In [7]:
df_hourly = (df_hourly
             .join(df_hourly.groupby(['Hour','Weekday'])['value'].mean(),
                   on = ['Hour', 'Weekday'], rsuffix='_Average')
            )

df_daily = (df_daily
            .join(df_daily.groupby(['Hour','Weekday'])['value'].mean(),
                  on = ['Hour', 'Weekday'], rsuffix='_Average')
           )

df_hourly.tail()

C:\Users\xz159\AppData\Local\Temp\ipykernel_9520\3887481213.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .join(df_hourly.groupby(['Hour','Weekday'])['value'].mean(),
C:\Users\xz159\AppData\Local\Temp\ipykernel_9520\3887481213.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .join(df_daily.groupby(['Hour','Weekday'])['value'].mean(),


,timestamp,Unnamed: 0,value,Weekday,Hour,Day,Month,Year,Month_day,Lag,Rolling_Mean,value_Average
5155,2015-01-31 19:00:00,10310.5,28288.5,Saturday,19,5,1,2015,31,26665.0,23537.214286,24501.870968
5156,2015-01-31 20:00:00,10312.5,24138.0,Saturday,20,5,1,2015,31,28288.5,23673.571429,22193.758065
5157,2015-01-31 21:00:00,10314.5,24194.5,Saturday,21,5,1,2015,31,24138.0,24031.214286,21983.241935
5158,2015-01-31 22:00:00,10316.5,26515.0,Saturday,22,5,1,2015,31,24194.5,24635.714286,23949.951613
5159,2015-01-31 23:00:00,10318.5,26439.5,Saturday,23,5,1,2015,31,26515.0,25485.071429,25192.516129


In [8]:
#Clear nulls
df_hourly.dropna(inplace=True)

# Daily
df_daily_model_data = df_daily[['value', 'Hour', 'Day',  'Month','Month_day','Rolling_Mean']].dropna()

# Hourly
model_data = df_hourly[['value', 'Hour', 'Day', 'Month_day', 'Month','Rolling_Mean','Lag', 'timestamp']].set_index('timestamp').dropna()
model_data.head()

,value,Hour,Day,Month_day,Month,Rolling_Mean,Lag
timestamp,,,,,,,
2014-07-01 01:00:00,5433.0,1,1,1,7,7459.250000,9485.5
2014-07-01 02:00:00,3346.5,2,1,1,7,6088.333333,5433.0
2014-07-01 03:00:00,2216.5,3,1,1,7,5120.375000,3346.5
2014-07-01 04:00:00,2189.5,4,1,1,7,4534.200000,2216.5
2014-07-01 05:00:00,3439.5,5,1,1,7,4351.750000,2189.5


In [13]:
from sklearn.ensemble import IsolationForest

def run_isolation_forest(model_data: pd.DataFrame, contamination=0.005, n_estimators=200, max_samples=0.7) -> pd.DataFrame:
    
    IF = (IsolationForest(random_state=0,
                          contamination=contamination,
                          n_estimators=n_estimators,
                          max_samples=max_samples)
         )
    
    IF.fit(model_data)
    
    output = pd.Series(IF.predict(model_data)).apply(lambda x: 1 if x == -1 else 0)
    
    score = IF.decision_function(model_data)
    
    return output, score

In [14]:
outliers, score = run_isolation_forest(model_data)

In [15]:
df_hourly = (df_hourly
             .assign(Outliers = outliers)
             .assign(Score = score)
            )

df_hourly

,timestamp,Unnamed: 0,value,Weekday,Hour,Day,Month,Year,Month_day,Lag,Rolling_Mean,value_Average,Outliers,Score
1,2014-07-01 01:00:00,2.5,5433.0,Tuesday,1,1,7,2014,1,9485.5,7459.250000,5028.193548,0.0,0.037801
2,2014-07-01 02:00:00,4.5,3346.5,Tuesday,2,1,7,2014,1,5433.0,6088.333333,3052.112903,0.0,0.056089
3,2014-07-01 03:00:00,6.5,2216.5,Tuesday,3,1,7,2014,1,3346.5,5120.375000,2039.580645,0.0,0.053583
4,2014-07-01 04:00:00,8.5,2189.5,Tuesday,4,1,7,2014,1,2216.5,4534.200000,2031.258065,0.0,0.061102
5,2014-07-01 05:00:00,10.5,3439.5,Tuesday,5,1,7,2014,1,2189.5,4351.750000,3207.338710,0.0,0.076468
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5155,2015-01-31 19:00:00,10310.5,28288.5,Saturday,19,5,1,2015,31,26665.0,23537.214286,24501.870968,0.0,0.001273
5156,2015-01-31 20:00:00,10312.5,24138.0,Saturday,20,5,1,2015,31,28288.5,23673.571429,22193.758065,0.0,0.004279
5157,2015-01-31 21:00:00,10314.5,24194.5,Saturday,21,5,1,2015,31,24138.0,24031.214286,21983.241935,0.0,0.046274
5158,2015-01-31 22:00:00,10316.5,26515.0,Saturday,22,5,1,2015,31,24194.5,24635.714286,23949.951613,1.0,0.027190


In [16]:
def make_features(df, prefer_value_average=True):
    # 优先用 value_Average，没有就用 value
    val_col = 'value_Average' if prefer_value_average and 'value_Average' in df.columns else (
        'value' if 'value' in df.columns else None
    )
    base_feats = ['Hour','Day','Month_day','Month','Rolling_Mean','Lag']
    feats = ([val_col] if val_col else []) + [c for c in base_feats if c in df.columns]

    if not feats:
        raise ValueError('一个可用特征都没有。先把衍生列算出来：Hour/Day/Month_day/Month/Lag/Rolling_Mean。')

    X = df[feats].dropna()
    print('使用特征：', feats, '  形状：', X.shape)
    return X

X = make_features(df_hourly)


使用特征： ['value_Average', 'Hour', 'Day', 'Month_day', 'Month', 'Rolling_Mean', 'Lag']   形状： (5159, 7)


In [18]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

# 假设你有标签 Outliers（0/1），没有就跳过评估这块
y = df_hourly.loc[X.index, 'Outliers'].fillna(0).astype(int) if 'Outliers' in df_hourly.columns else None

# 按时间/顺序切 8:2（索引本来就是按时间升序的）
cut = int(len(X)*0.8)
X_train, X_test = X.iloc[:cut], X.iloc[cut:]
y_test = y.iloc[cut:] if y is not None else None

IF = IsolationForest(
    random_state=0,
    contamination=0.005,
    n_estimators=300,
    max_samples=256
)
IF.fit(X_train)

scores = IF.decision_function(X_test)  # 分数越小越异常
pred   = (IF.predict(X_test) == -1).astype(int)

if y_test is not None:
    roc = roc_auc_score(y_test, -scores)  # 方向取负
    pr  = average_precision_score(y_test, -scores)
    print(f'ROC-AUC={roc:.4f}  PR-AUC={pr:.4f}')


ROC-AUC=0.9561  PR-AUC=0.3919


In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())


2.5.1+cu121
12.1
True


In [26]:
import sys, torch, pathlib
print("KERNEL PY =", sys.executable)
print("torch in memory =", torch.__version__, torch.__file__)


KERNEL PY = d:\Software\anaconda\envs\datascience\python.exe
torch in memory = 2.8.0+cpu d:\Software\anaconda\envs\datascience\Lib\site-packages\torch\__init__.py


In [17]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# ==== 1) 准备特征 ====
FEATS = ['value','Hour','Day','Month_day','Month','Rolling_Mean','Lag']
X_all = df_hourly[FEATS].dropna()                 # 去掉 rolling/lag 产生的 NaN

# （可选）标签列名；没有就报错
LABEL_COL = 'Outliers'
assert LABEL_COL in df_hourly.columns, f'缺少标签列 {LABEL_COL}'
y_all = df_hourly.loc[X_all.index, LABEL_COL].fillna(0).astype(int)

# ==== 2) 按时间切分（8:2，防止信息泄露）====
cut = int(len(X_all) * 0.8)
X_train_df, X_test_df = X_all.iloc[:cut], X_all.iloc[cut:]
y_train,   y_test     = y_all.iloc[:cut], y_all.iloc[cut:]

# ==== 3) 转 numpy（float32）====
X_train = X_train_df.to_numpy(dtype=np.float32)
X_test  = X_test_df.to_numpy(dtype=np.float32)

# ==== 4) 跑 DIF ====
from algorithms import DIF

dif = DIF(                      # 别传 contamination 到 DIF！
    n_ensemble=50,
    n_estimators=6,           # 6 太少了，100 起步更稳
    random_state=0
)

# 如果你的 DIF 实现支持内部 IF 参数，就这样设置；没有就删掉这段
if hasattr(dif, 'if_params'):
    dif.if_params.update(dict(contamination=0.005, max_samples=0.7))
elif hasattr(dif, 'set_if_params'):
    dif.set_if_params(contamination=0.005, max_samples=0.7)

dif.fit(X_train)

# 连续分数（通常“越小越异常”）
score = np.asarray(dif.decision_function(X_test), dtype=float)

# ==== 5) 计算 ROC-AUC / PR-AUC ====
# 大多数 DIF/IF：分数小=更异常；方向不对自动翻转一次
roc = roc_auc_score(y_test, score)
if roc < 0.5:
    score = -score
    roc = roc_auc_score(y_test, score)

pr  = average_precision_score(y_test, score)

print(f'样本数：train={len(X_train)}, test={len(X_test)}, 正类(异常)比例={y_test.mean():.4f}')
print(f'ROC-AUC={roc:.4f}  PR-AUC={pr:.4f}')


network additional parameters: {'n_hidden': [500, 100], 'n_emb': 20, 'skip_connection': None, 'dropout': None, 'activation': 'tanh', 'be_size': 50}
样本数：train=4127, test=1032, 正类(异常)比例=0.0233
ROC-AUC=0.7154  PR-AUC=0.4852
